# TabICLv2 sobre las variables del modelo 15b

[TabICLv2](https://github.com/soda-inria/tabicl) (Inria) es un modelo fundacional para datos tabulares. No se entrena con cada dataset: recibe las filas de entrenamiento como contexto y predice las nuevas en una sola pasada de un transformer preentrenado (*in-context learning*). Funciona de forma muy distinta a los árboles. Por eso es un candidato para el ensamble: en el modelo 15 las combinaciones fallaron porque LightGBM y CatBoost se parecen demasiado.

**Datos** (dataset `tabicl_datos`, exportado por `kaggle/exportar_tabicl_datos.py`): las 147 variables de la variante 15b (08 + promedio de los tres fine-tunings + 22 patrones), en float32, con las categóricas como `category` (TabICL las codifica solo) y los NaN como NaN.
- **Anidada** (para evaluar): para cada fold externo k, `anidada_fold{k}_train` (los otros 4 folds, con las variables con target calculadas con los folds internos) y `anidada_fold{k}_valid` (el fold k). Son exactamente las matrices que usó el LightGBM 15b en la validación anidada (0,9458).
- **Estándar** (para test): `estandar_train` (70.000 filas, variables con target out-of-fold) y `estandar_test` (30.000 filas, con todo train).

**Configuraciones**, una detrás de otra:

| configuración | variables | n_estimators |
|---|---|---|
| `todas` | las 147 | 8 |
| `top100` | las 100 más importantes del LightGBM 15b (TabICLv2 se preentrenó con hasta 100 columnas) | 8 |
| `top100_n16` | las mismas 100 | 16; solo si el primer fold de `todas` tarda menos de 5 minutos |

**Qué hace cada configuración:**
1. Para cada fold k: `fit` con el entrenamiento anidado y `predict_proba` sobre la validación del fold. Guarda el fold en `/kaggle/working` apenas termina y muestra accuracy, AUC, log loss y tiempo.
2. Test: `fit` con `estandar_train` y `predict_proba` sobre `estandar_test`.
3. Salidas: `tabicl_{config}_oof.csv` (fila, fold, proba_used) y `tabicl_{config}_test.csv` (fila de test desde 0, proba_used).
4. Al final, una tabla compara las configuraciones con el LightGBM 15b.

**Memoria:** si la GPU se queda sin memoria, reintenta con `batch_size` 4 y después 2 (miembros del ensamble procesados a la vez), con `offload_mode="auto"`, y lo informa.

**Retomar:** los folds y los test ya guardados en `/kaggle/working` se saltean.

**Cuánto tarda** (GPU T4): no hay una medición previa en este dataset (56.000 filas de contexto y 14.000 a predecir por fold).
- Estimado: **~5-30 minutos por fold**. El test cuesta ~2-3 folds, porque tiene más contexto (70.000) y más filas (30.000).
- Total para `todas` + `top100`: **~1-7 horas**.
- Después del primer fold se muestra una estimación con el tiempo medido y un aviso si supera `LIMITE_HORAS`.

In [ ]:
# ---- Parámetros ----
CONFIGURACIONES = [
    {"nombre": "todas", "columnas": "todas", "n_estimators": 8},
    {"nombre": "top100", "columnas": "top100", "n_estimators": 8},
]
# Se agrega si el primer fold de "todas" tarda menos de MINUTOS_EXTRA minutos
EXTRA = {"nombre": "top100_n16", "columnas": "top100", "n_estimators": 16}
MINUTOS_EXTRA = 5
SEMILLA = 42
DISPOSITIVO = "cuda"
REQUERIR_GPU = True
INSTALAR = True  # pip install -U tabicl (la versión más reciente)
RUTA_ENTRADA = "/kaggle/input"
RUTA_SALIDA = "/kaggle/working"
RUTA_OFFLOAD = "/tmp/tabicl_offload"  # por si offload_mode="auto" tiene que pasar a disco
MAX_FILAS = None  # solo para pruebas: limita las filas de cada archivo
LIMITE_HORAS = 8.5  # una sesión de Kaggle dura como máximo ~9 horas
N_FOLDS = 5

## Instalación y GPU

In [ ]:
import subprocess
import sys

if INSTALAR:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "tabicl"], check=True)

import gc
import glob
import json
import os
import time
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from tabicl import TabICLClassifier

print(f"tabicl {version('tabicl')} | torch {torch.__version__}")
hay_gpu = torch.cuda.is_available()
if hay_gpu:
    propiedades = torch.cuda.get_device_properties(0)
    print(f"GPU: {propiedades.name} | memoria {propiedades.total_memory / 1024 ** 3:.1f} GB"
          + (f" | {torch.cuda.device_count()} GPU (se usa la primera)" if torch.cuda.device_count() > 1 else ""))
else:
    print("No hay GPU")
if REQUERIR_GPU and not hay_gpu:
    raise RuntimeError("No hay GPU. En el panel derecho del notebook: Settings > Accelerator > GPU T4 x2 (o GPU T4), "
                       "y volvé a ejecutar. TabICL en CPU tardaría muchas horas.")
os.makedirs(RUTA_SALIDA, exist_ok=True)
INICIO_CORRIDA = time.time()

## Datos

Se buscan los archivos con `glob` en cualquier subcarpeta de `/kaggle/input`. Al leer cada parquet se vuelven a aplicar las categorías de train guardadas en `columnas.json`: pyarrow lee como texto las categóricas escritas con fastparquet.

In [ ]:
def buscar(nombre):
    encontrados = sorted(glob.glob(os.path.join(RUTA_ENTRADA, "**", nombre), recursive=True))
    if not encontrados:
        contenido = sorted(glob.glob(os.path.join(RUTA_ENTRADA, "**", "*"), recursive=True))[:100]
        raise FileNotFoundError(
            f"No se encontró {nombre} dentro de {RUTA_ENTRADA}. Revisá que el dataset tabicl_datos esté agregado "
            f"como dato de entrada (panel derecho: Input > Add Input). Lo que hay ahora:\n"
            + ("\n".join(contenido) if contenido else "(vacío)"))
    if len(encontrados) > 1:
        print(f"Aviso: hay {len(encontrados)} archivos {nombre}; se usa {encontrados[0]}")
    return encontrados[0]


with open(buscar("columnas.json"), encoding="utf-8") as f:
    COLUMNAS = json.load(f)
CARPETA_DATOS = os.path.dirname(buscar("columnas.json"))
TIPOS = {c: pd.CategoricalDtype(categorias) for c, categorias in COLUMNAS["categorias"].items()}
LGBM_15B = COLUMNAS["lgbm_15b_accuracy_anidada"]
assert len(COLUMNAS["todas"]) == 147 and len(COLUMNAS["top100"]) == 100 and set(COLUMNAS["top100"]) <= set(COLUMNAS["todas"])


def leer(nombre):
    datos = pd.read_parquet(buscar(f"{nombre}.parquet"))
    for c, tipo in TIPOS.items():
        nulos = datos[c].isna().sum()
        datos[c] = datos[c].astype(object).where(datos[c].notna(), None).astype(tipo)
        assert datos[c].isna().sum() == nulos, f"{nombre}: {c} tiene valores fuera de las categorías de train"
    assert list(datos.columns[:147]) == COLUMNAS["todas"], f"{nombre}: columnas distintas a columnas.json"
    return datos.head(MAX_FILAS) if MAX_FILAS else datos


print(f"Carpeta de datos: {CARPETA_DATOS}")
ejemplo = leer("anidada_fold1_train")
print(f"anidada_fold1_train: {ejemplo.shape[0]:,} filas | {(ejemplo.dtypes == 'category').sum()} categóricas | "
      f"tipos: {ejemplo[COLUMNAS['todas']].dtypes.astype(str).value_counts().to_dict()} | % used {ejemplo['y'].mean():.3f}")
del ejemplo

## Modelo

`TabICLClassifier(device="cuda", random_state=42, n_estimators=...)`. Si hay un error de memoria, se reintenta con `batch_size=4` y después `batch_size=2`, con `offload_mode="auto"`; el intento usado queda guardado en el archivo de información de cada fold.

In [ ]:
INTENTOS = [{}, {"batch_size": 4, "offload_mode": "auto"}, {"batch_size": 2, "offload_mode": "auto"}]


def es_error_memoria(error):
    return isinstance(error, torch.cuda.OutOfMemoryError) or "out of memory" in str(error).lower()


def ajustar_predecir(X_ajuste, y_ajuste, X_aplicar, n_estimators):
    # Devuelve (probabilidad de used, descripción del intento que funcionó)
    for i, extra in enumerate(INTENTOS):
        modelo = None
        try:
            modelo = TabICLClassifier(device=DISPOSITIVO, random_state=SEMILLA, n_estimators=n_estimators,
                                      disk_offload_dir=RUTA_OFFLOAD if extra else None, **extra)
            modelo.fit(X_ajuste, y_ajuste)
            proba = modelo.predict_proba(X_aplicar)[:, list(modelo.classes_).index(1)]
            return proba, (str(extra) if extra else "por defecto (batch_size 8)")
        except Exception as error:
            if not es_error_memoria(error) or i == len(INTENTOS) - 1:
                raise
            print(f"    Error de memoria con {extra or 'los parámetros por defecto'}: {str(error)[:150]}")
            print(f"    Reintento con {INTENTOS[i + 1]}")
        finally:
            del modelo
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


def metricas(y, proba):
    proba = np.clip(proba, 1e-6, 1 - 1e-6)
    return {"accuracy": accuracy_score(y, proba >= 0.5), "auc": roc_auc_score(y, proba), "log_loss": log_loss(y, proba)}


def archivo(*partes):
    return os.path.join(RUTA_SALIDA, "tabicl_" + "_".join(str(p) for p in partes))

## Evaluación anidada y test

Cada fold se guarda en `tabicl_{config}_fold_{k}.csv` y su información (métricas, tiempo, intento) en `tabicl_{config}_fold_{k}.json`. Si ya existen, el fold se saltea. Con los 5 folds se arma `tabicl_{config}_oof.csv`; después se ajusta con todo train y se predice test (`tabicl_{config}_test.csv`).

In [ ]:
def estimar(minutos_fold, configuraciones, hechas):
    # Folds y tests que faltan; el test (70.000 de contexto, 30.000 a predecir) ~2,5 folds; n_estimators escala lineal
    restante = 0.0
    for cfg in configuraciones:
        escala = cfg["n_estimators"] / 8
        for k in range(1, N_FOLDS + 1):
            if not os.path.exists(archivo(cfg["nombre"], "fold", k) + ".csv") and (cfg["nombre"], k) not in hechas:
                restante += minutos_fold * escala
        if not os.path.exists(archivo(cfg["nombre"], "test.csv")):
            restante += 2.5 * minutos_fold * escala
    return restante


def correr_configuracion(cfg, configuraciones):
    nombre, columnas = cfg["nombre"], COLUMNAS[cfg["columnas"]]
    print(f"\n===== {nombre}: {len(columnas)} variables, n_estimators = {cfg['n_estimators']} =====")
    partes, hechas = [], set()
    for k in range(1, N_FOLDS + 1):
        ruta = archivo(nombre, "fold", k)
        if os.path.exists(ruta + ".csv") and os.path.exists(ruta + ".json"):
            with open(ruta + ".json", encoding="utf-8") as f:
                info = json.load(f)
            print(f"Fold {k}: ya estaba guardado (accuracy {info['accuracy']:.4f}, {info['minutos']:.1f} min)")
        else:
            entrenamiento, validacion = leer(f"anidada_fold{k}_train"), leer(f"anidada_fold{k}_valid")
            inicio = time.time()
            proba, intento = ajustar_predecir(entrenamiento[columnas], entrenamiento["y"], validacion[columnas],
                                              cfg["n_estimators"])
            minutos = (time.time() - inicio) / 60
            pd.DataFrame({"fila": validacion["fila"], "fold": k, "proba_used": proba}).to_csv(ruta + ".csv", index=False)
            info = {**metricas(validacion["y"], proba), "minutos": minutos, "intento": intento,
                    "filas_ajuste": len(entrenamiento), "filas_validacion": len(validacion)}
            with open(ruta + ".json", "w", encoding="utf-8") as f:
                json.dump(info, f)
            hechas.add((nombre, k))
            print(f"Fold {k}: accuracy {info['accuracy']:.4f} | AUC {info['auc']:.4f} | log loss {info['log_loss']:.4f} | "
                  f"{minutos:.1f} min | intento: {intento}")
            if len(hechas) == 1:
                restante = estimar(minutos / (cfg["n_estimators"] / 8), configuraciones, hechas)
                transcurrido = (time.time() - INICIO_CORRIDA) / 3600
                print(f"  Estimación: faltan ~{restante / 60:.1f} h para las configuraciones de la lista "
                      f"(total de la sesión ~{transcurrido + restante / 60:.1f} h)")
                if transcurrido + restante / 60 > LIMITE_HORAS:
                    print(f"  AVISO: supera {LIMITE_HORAS} h. Lo guardado no se pierde: se puede retomar en otra sesión "
                          f"(ver INSTRUCCIONES_KAGGLE.md) o quitar una configuración.")
        partes.append(pd.read_csv(ruta + ".csv"))
        if nombre == "todas" and k == 1 and EXTRA not in configuraciones:
            if info["minutos"] < MINUTOS_EXTRA:
                configuraciones.append(EXTRA)
                print(f"  El fold 1 de 'todas' tardó {info['minutos']:.1f} min (< {MINUTOS_EXTRA}): "
                      f"se agrega la configuración '{EXTRA['nombre']}'")
            else:
                print(f"  El fold 1 de 'todas' tardó {info['minutos']:.1f} min (>= {MINUTOS_EXTRA}): "
                      f"no se agrega '{EXTRA['nombre']}'")
    oof = pd.concat(partes).sort_values("fila").reset_index(drop=True)
    oof.to_csv(archivo(nombre, "oof.csv"), index=False)

    ruta_test = archivo(nombre, "test.csv")
    if os.path.exists(ruta_test):
        print(f"Test: ya estaba guardado ({ruta_test})")
    else:
        train, test = leer("estandar_train"), leer("estandar_test")
        inicio = time.time()
        proba, intento = ajustar_predecir(train[columnas], train["y"], test[columnas], cfg["n_estimators"])
        pd.DataFrame({"fila": test["fila"], "proba_used": proba}).to_csv(ruta_test, index=False)
        print(f"Test: {len(test):,} filas, contexto de {len(train):,} | {(time.time() - inicio) / 60:.1f} min | "
              f"intento: {intento} | % used predicho {np.mean(proba >= 0.5):.3f}")
    return oof


configuraciones = [dict(c) for c in CONFIGURACIONES]
resultados = {}
i = 0
while i < len(configuraciones):  # la lista puede crecer con EXTRA
    cfg = configuraciones[i]
    resultados[cfg["nombre"]] = correr_configuracion(cfg, configuraciones)
    i += 1
print(f"\nTiempo total de esta sesión: {(time.time() - INICIO_CORRIDA) / 60:.1f} min")

## Comparación con el LightGBM 15b

Métricas OOF de la validación anidada (las mismas filas y los mismos folds). El LightGBM 15b tiene 0,9458 de accuracy anidada. Lo que importa para el ensamble no es superarlo solo, sino que sus errores sean distintos: la correlación y la combinación se calculan después, localmente, en el notebook del proyecto.

In [ ]:
filas = {}
for nombre, oof in resultados.items():
    y_oof = pd.concat([leer(f"anidada_fold{k}_valid")[["fila", "y"]] for k in range(1, N_FOLDS + 1)])
    unido = oof.merge(y_oof, on="fila", how="left")
    assert unido["y"].notna().all()
    por_fold = [metricas(g["y"], g["proba_used"])["accuracy"] for _, g in unido.groupby("fold")]
    filas[nombre] = {**metricas(unido["y"], unido["proba_used"].to_numpy()),
                     "accuracy media por fold": np.mean(por_fold), "desvío": np.std(por_fold, ddof=1),
                     "diferencia con LightGBM 15b (puntos)": (np.mean(por_fold) - LGBM_15B) * 100}
tabla = pd.DataFrame(filas).T
tabla.loc["LightGBM 15b (referencia)", ["accuracy media por fold"]] = LGBM_15B
display(tabla.round(4))
print("Archivos para descargar:")
for ruta in sorted(glob.glob(os.path.join(RUTA_SALIDA, "tabicl_*_oof.csv")) + glob.glob(os.path.join(RUTA_SALIDA, "tabicl_*_test.csv"))):
    print(f"  {ruta}")